## Bijdrage Raoul Buffels — DeepLearningTeam10

Deze Citi Bike-uitwerking is aangeleverd als de bijdrage van **Raoul Buffels**. Teamleden: Jorre Van Dyck, Milan Wouters, Raoul Buffels en Andrew Noeyens. De concrete menselijke taakverdeling moet het team zelf aanvullen. **AI-gebruik:** Codex hielp met implementatie, uitleg, tests, uitvoering en verpakking.

De codecellen zijn uitgevoerd in de oorspronkelijke lokale `Citybike_full`-omgeving op 9 oktober 2026. Deze export behoudt die echte uitvoer; alleen deze Markdown-cel is toegevoegd. Lokale paden, timings en fingerprints in de rapporten beschrijven de oorspronkelijke uitvoering. Start voor heruitvoering vanuit `RaoulBuffels` of de map `notebooks` met de bijbehorende Python 3.11-omgeving. AWS is een afzonderlijke latere taak.

# 04 — PyCaret met dezelfde chronologische folds

Bijdragen: door team in te vullen. De setup gebruikt expliciete validatie, opgeslagen folds en preprocess=False. Ridge schaalt binnen de eigen fold-pipeline. Alle estimators volgen dezelfde niet-negatieve en zomertijdregels.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'Citybike_full']
ROOT = next((p for p in locations if (p / 'citibike').exists()), None)
if ROOT is None: raise RuntimeError('Start dit notebook vanuit Citybike_full of notebooks.')
sys.path.insert(0, str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
from citibike.datasets import manifest
from citibike.plots import show
def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS / name, dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


De uitgevoerde setup in `citibike.modeling.py` gebruikt onderstaande instellingen. `compare_models` ontvangt de vier gedeelde estimators, inclusief de operationele clipping en zomertijdregel. De audit controleert de daadwerkelijke interne train- en validatierijen.

```python
experiment.setup(data=train[FEATURES+["rides"]],
    test_data=validation[FEATURES+["rides"]], target="rides",
    fold_strategy=SavedChronologicalCV(bounds), fold=len(bounds),
    data_split_shuffle=False, fold_shuffle=False, preprocess=False,
    session_id=42, n_jobs=2, index=False)
```

In [2]:
table('pycaret_comparison.csv')
display(pd.DataFrame(report('pycaret_split_audit.json')['folds']))

,Unnamed: 0,candidate,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
0,3,lightgbm,Light Gradient Boosting Machine,919.4277,2.219320e+06,1433.3589,0.7183,0.8604,1.0487,1.208
1,2,random_forest,Random Forest Regressor,917.6491,2.280610e+06,1460.7833,0.7014,0.8548,1.1461,51.088
2,1,ridge,Ridge Regression,1025.6297,2.668287e+06,1594.3144,0.6369,0.9592,1.1846,0.118
3,0,seasonal_naive,SeasonalNaive,1349.9824,4.550459e+06,2086.6729,0.3707,1.1928,1.2417,0.710


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30


In [3]:
audit=report('pycaret_split_audit.json')
display(pd.DataFrame([{k:v for k,v in audit.items() if k!='folds'}]))

,train_rows,validation_rows,fold_fingerprint,train_data_fingerprint,validation_data_fingerprint,shuffle,preprocess,preprocessing,actual_internal_rows_checked,candidate_labels,operational_prediction_rules
0,112872,1464,59238f467d526664e56943f96d102928ede96e4e371fec...,f5aec178a9fea0223f2d08c98b207909faf272d8a43ff1...,83d6332e2bdba4915e1ee1b4f753468e8fe682bc11feb2...,False,False,Ridge scaler inside per-fold estimator pipeline,True,"[lightgbm, random_forest, ridge, seasonal_naive]",same OperationalRegressor as sklearn and saved...


PyCaret verkleint numerieke dtypes bij de setup. Daardoor kunnen afronding en de binning van LightGBM iets verschillen, ondanks dezelfde informatie en folds. De onderstaande tabel toont het gemeten verschil. De uiteindelijke selectie gebruikt de validatiemetrics van de handmatige, gedeelde inference-pipeline; AutoML is een aanvullende vergelijking.

In [4]:
manual=pd.DataFrame(report('model_comparison.json'))[['name','cv_RMSE']]
automl=pd.read_csv(REPORTS/'pycaret_comparison.csv')
precision=automl[['candidate','RMSE']].merge(manual,left_on='candidate',right_on='name')
precision['CV_RMSE_verschil']=precision.RMSE-precision.cv_RMSE
display(precision)

,candidate,RMSE,name,cv_RMSE,CV_RMSE_verschil
0,lightgbm,1433.3589,lightgbm,1423.037130,10.321770
1,random_forest,1460.7833,random_forest,1460.783347,-0.000047
2,ridge,1594.3144,ridge,1594.314667,-0.000267
3,seasonal_naive,2086.6729,seasonal_naive,2086.672928,-0.000028
